# Bengali Handwritten OCR Fine-Tuning with Qwen2-VL (QLoRA) on Kaggle
Train on the BN-HTRd dataset using 4-bit NF4 QLoRA on Kaggle Dual Tesla T4 GPUs (30 hours/week free).

> **Important Settings** in the right-hand panel:
- **Accelerator**: `GPU T4 x2`
- **Internet**: `Internet on`
- **Persistence**: `Variables and Files`

In [ ]:
# 1. Verify Dual GPU availability
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU found! Enable GPU T4 x2 in Session Options."
print(f"CUDA Device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 2. Install required packages (Notice: torch is pre-installed with CUDA)
!pip install -q "transformers>=4.45.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" "qwen-vl-utils>=0.0.8" datasets pandas openpyxl pillow jiwer gdown

In [ ]:
# 3. Setup workspace & clone repository
import os
%cd /kaggle/working
if not os.path.exists('/kaggle/working/bangla-ocr-qwen2vl'):
    !git clone https://github.com/RahulIslam46/bangla-ocr-qwen2vl.git /kaggle/working/bangla-ocr-qwen2vl
%cd /kaggle/working/bangla-ocr-qwen2vl
!git pull
!mkdir -p /kaggle/working/bangla_ocr_checkpoints

In [ ]:
# 4. Check dataset location
import os, glob

# Find dataset directory in Kaggle inputs or local working directory
data_candidates = glob.glob('/kaggle/input/**/Dataset', recursive=True) + glob.glob('/kaggle/working/**/Dataset', recursive=True)
if data_candidates:
    data_dir = data_candidates[0]
else:
    data_dir = '/kaggle/input/bn-htrd/Dataset'

print(f"Target dataset directory: {data_dir}")
print(f"Directory exists: {os.path.exists(data_dir)}")

In [ ]:
# 5. Launch Training
# Runs 4-bit QLoRA on BN-HTRd dataset
!python train.py \
    --data_dir "$data_dir" \
    --mode line \
    --output_dir /kaggle/working/bangla_ocr_checkpoints \
    --epochs 3 \
    --batch_size 1 \
    --grad_accum 8 \
    --lr 2e-4 \
    --save_steps 50 \
    --logging_steps 5

In [ ]:
# 6. Resume training from checkpoint (if needed)
# !python train.py \
#     --data_dir "$data_dir" \
#     --mode line \
#     --output_dir /kaggle/working/bangla_ocr_checkpoints \
#     --resume_from_checkpoint /kaggle/working/bangla_ocr_checkpoints/checkpoint-350

In [ ]:
# 7. Evaluate fine-tuned model
!python evaluate.py \
    --data_dir "$data_dir" \
    --adapter_path /kaggle/working/bangla_ocr_checkpoints/final_model \
    --num_samples 20